In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [6]:
class State_(TypedDict):
    topic: str
    outline: str
    response: str

In [7]:

llm = HuggingFaceEndpoint(
    repo_id="google/gemma-4-31B-it",
    task="text-generation"
)
chat = ChatHuggingFace(llm=llm)

In [8]:
def outline_generator (state: State_) -> State_:
    topic = state['topic']
    prompt = PromptTemplate(
        template="give me the outline of the given {topic}",
        input_variables=['topic']
    )
    response = prompt.invoke({'topic' : topic})

    result = chat.invoke(response)

    state['outline'] = result.content

    return state


In [9]:
def essay_generator (state: State_) -> State_:
    topic = state['topic']
    outline = state['outline']
    prompt = PromptTemplate(
            template="give me the eassy of 100 words on the given {topic} and the outline for the {topic} is {outline}",
            input_variables=['topic', 'outline']
        )
    fprompt = prompt.invoke({'topic': topic, 'outline':outline})

    result = chat.invoke(fprompt)
    state['response'] = result.content

    return state

In [10]:
# define state

graph = StateGraph(State_)

graph.add_node('outline_generator', outline_generator)
graph.add_node('essay_generator', essay_generator)

graph.add_edge(START, 'outline_generator')
graph.add_edge('outline_generator', 'essay_generator')
graph.add_edge('essay_generator', END)

workflow = graph.compile()

In [11]:
res = workflow.invoke({'topic': 'what is living a life means'})

res

{'topic': 'what is living a life means',
 'outline': 'Since you didn\'t provide a specific text, I have created a **comprehensive general outline** that explores the concept of "what living a life means." \n\nDepending on whether you are writing an essay, giving a speech, or journaling, you can choose the angle that fits best (Philosophical, Practical, or Emotional).\n\n---\n\n# Title:',
 'response': 'Since you didn\'t provide a specific title for your outline, I have written a reflective and concise essay based on the general concept of "What Living a Life Means."\n\n**Title: Beyond Existence: The Meaning of Living**\n\nLiving a life is far more than the biological act of existing; it is the conscious art of engaging with the world. While existing is a passive state, "living" is an active choice. It means embracing the full spectrum of human experience—celebrating triumphs and learning from failures. To truly live is to pursue passions, cultivate deep connections, and remain curious. 